## Tech Challenge FIAP - Fase 3

Grupo_37 -  MACHINE LEARNING ENGINEERING

Integrantes:
* D?bora Correia Campos
* Diangelis Barbato Fran?a
* Fabr?cio Zambom Galvani
* Vinicius Meira Albuquerque

### Bibliotecas

As principais bibliotecas como pandas, matplotlib, json, numpy e datetime s?o carregadas para trabalhar com os dados, manipula??o e visualiza??o de gr?ficos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import json
import os
import numpy as np
import datetime
from datetime import datetime
import boto3
import sagemaker

### Leitura dos dados

O arquivo CSV contendo os dados hist?ricos das a??es da Petrobras (PETR4.SA.csv) ? carregado para um DataFrame base.

In [ ]:
# Aqui, o arquivo CSV ? recarregado, agora convertendo a coluna Date em objetos datetime e definindo essa coluna como o ?ndice do DataFrame.
dateparse = lambda dates: datetime.strptime(dates, '%Y-%m-%d')
base = pd.read_csv('PETR4.SA.csv', parse_dates = ['Date'], index_col = 'Date', date_parser = dateparse)
base

### Descri??o dos dados

Neste dataset apresentaremos um conjunto de dados extra?dos de cota??es de a??es da Petrobras (PETR4.SA) dispon?vel em: https://www.kaggle.com/datasets/delusdias/banco-do-brazil-sa-bbas3sa

Conte?do:

-Date: data da cota??o;

-Open: valor de abertura;

-High: maior valor;

-Low: menor valor;

-Close: valor de fechamento;

-Adj Close: valor de fechamento ajustado;

-Volume: quantidade de a??es ou contratos que foram negociados

### Prepara??o

**Remo??o de colunas desnecess?rias**: Apenas a coluna Close (fechamento da a??o) ? mantida, as outras colunas s?o descartadas, como Open, High, Low

In [ ]:
base.drop(columns = ['Open', 'High', 'Low', 'Adj Close', 'Volume'], axis = 1, inplace = True)
base

**C?pia do dataset**: Uma c?pia do DataFrame ? feita para criar uma base de teste, que ser? usada mais tarde para prever os valores das a??es.

In [ ]:
base_teste = base.copy()
base_teste.shape

**Divis?o dos dados de treinamento**: Seleciona as primeiras 1243 linhas do dataset para formar o conjunto de treinamento. Isso significa que voc? est? usando dados de um per?odo espec?fico para treinar o modelo

In [ ]:
base_treinamento = base.iloc[0:1243:]
base_treinamento

**Defini??o de datas**: Estabelece as datas de in?cio e fim para o conjunto de treinamento e a data m?xima para previs?o.

In [ ]:
base_treinamento.index.min(), base_treinamento.index.max()

In [ ]:
base_teste.index.min(), base_teste.index.max()

In [ ]:
dt_data_inicial = pd.Timestamp("2014-05-19")
dt_data_final_treinamento = pd.Timestamp("2019-04-03")
dt_previsao_maxima = pd.Timestamp("2019-05-16")

**Intervalo de datas**: Cria intervalos de datas para o per?odo de treinamento e teste.

In [ ]:
faixa_treinamento = pd.date_range(dt_data_inicial, dt_data_final_treinamento, freq='D')
faixa_teste = pd.date_range(dt_data_inicial, dt_previsao_maxima, freq='D')

In [ ]:
faixa_treinamento

In [ ]:
faixa_teste

**Cria??o de s?ries temporais**: Cria duas listas, uma para armazenar os valores de fechamento do per?odo de treinamento e outra para o teste.

In [ ]:
serie_temporal_treinamento = []
serie_temporal_teste = []

serie_temporal_treinamento.append(base_treinamento[faixa_treinamento[0]:faixa_treinamento[1214]]['Close'])
serie_temporal_teste.append(base_teste[faixa_teste[0]:faixa_teste[1243]]['Close'])

In [ ]:
serie_temporal_treinamento

In [ ]:
serie_temporal_teste

In [ ]:
len(serie_temporal_treinamento[0]), len(serie_temporal_teste[0])

**Plotagem das s?ries temporais**: Gr?fico que mostra o comportamento das s?ries de treinamento e teste para facilitar a visualiza??o.

In [ ]:
plt.figure(figsize=(10,10))
serie_temporal_teste[0].plot(label = "Teste")
serie_temporal_treinamento[0].plot(label = "Treinamento")
plt.legend()
plt.show()

**Fun??o encode_target**: Transforma a s?rie temporal em uma lista, substituindo valores n?o finitos (NaN) por strings "NaN"

In [ ]:
def encode_target(ts):
    return [x if np.isfinite(x) else "NaN" for x in ts]

In [ ]:
encode_target(serie_temporal_treinamento[0][0:5])

**Fun??o series_to_obj**: Converte uma s?rie temporal em um objeto JSON que cont?m a data de in?cio e os valores codificados

In [ ]:
def series_to_obj(ts):
    obj = {"start": str(ts.index[0]), "target": encode_target(ts)}
    return obj

In [ ]:
series_to_obj(serie_temporal_treinamento[0][0:5])

**Fun??o series_to_jsonline**: Serializa a s?rie temporal para o formato JSON.

In [ ]:
def series_to_jsonline(ts):
    return json.dumps(series_to_obj(ts))

In [ ]:
series_to_jsonline(serie_temporal_treinamento[0][0:5])

**Escrita dos dados**: As s?ries temporais de treinamento e teste s?o gravadas em arquivos .json no formato que ser? usado pelo Amazon Sagemaker.

In [ ]:
encoding = 'utf-8'
arquivo_json_treinamento = 'train_petr.json'
arquivo_json_teste = 'test_petr.json'
with open(arquivo_json_treinamento, 'wb') as fp:
    for ts in serie_temporal_treinamento:
        fp.write(series_to_jsonline(ts).encode(encoding))
        fp.write('\n'.encode(encoding))

with open(arquivo_json_teste, 'wb') as fp:
    for ts in serie_temporal_teste:
        fp.write(series_to_jsonline(ts).encode(encoding))
        fp.write('\n'.encode(encoding))

**Configura??o AWS**: Define as vari?veis para o treinamento no AWS Sagemaker, como bucket S3 para armazenar dados, nome do job e a fun??o

In [ ]:
session = sagemaker.Session()
role = sagemaker.get_execution_role()
job_name = 'deepar-petr'
bucket = 'grupo37fiapos'
subpasta_dataset = 'datasets/petr'
subpasta_modelo = 'modelos/petr/deepar'
s3_train_data = 's3://{}/{}/train/{}'.format(bucket, subpasta_dataset, 'train_petr.json')
s3_test_data = 's3://{}/{}/test/{}'.format(bucket, subpasta_dataset, 'test_petr.json')
output_location = 's3://{}/{}/output'.format(bucket, subpasta_modelo)

print('Role: {}'.format(role))
print('Localiza??o da base de treinamento: {}'.format(s3_train_data))
print('Localiza??o da base de teste: {}'.format(s3_test_data))
print('Modelo ser? salvo em: {}'.format(output_location))

**Upload de arquivos no S3**: Envia os arquivos train_petr.json e test_petr.json para o bucket S3 configurado

In [ ]:
def escreve_s3(arquivo, bucket, nome):
    with open(arquivo, 'rb') as f:
        return boto3.Session().resource('s3').Bucket(bucket).Object(nome).upload_fileobj(f)

In [ ]:
escreve_s3('train_petr.json', bucket, os.path.join(subpasta_dataset, 'train', 'train_petr.json'))
escreve_s3('test_petr.json', bucket, os.path.join(subpasta_dataset, 'test', 'test_petr.json'))

**Configura??o do modelo**: Cria??o do estimador para o DeepAR no Sagemaker, utilizando inst?ncias spot para reduzir custos.

In [ ]:
# https://docs.aws.amazon.com/sagemaker/latest/dg/ecr-sa-east-1.html
container = sagemaker.image_uris.retrieve(framework = 'forecasting-deepar', region = session.boto_region_name)

In [ ]:
# https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html
# https://github.com/aws-samples/amazon-sagemaker-managed-spot-training/blob/main/xgboost_built_in_managed_spot_training_checkpointing/xgboost_built_in_managed_spot_training_checkpointing.ipynb
deepar = sagemaker.estimator.Estimator(image_uri = container,
                                       role = role,
                                       instance_count = 1,
                                       instance_type = 'ml.m5.large',
                                       output_path = output_location,
                                       sagemaker_session = session,
                                       base_job_name = job_name,
                                       use_spot_instances = True,
                                       max_run = 3600,
                                       max_wait = 3600)

In [ ]:
# https://docs.aws.amazon.com/sagemaker/latest/dg/deepar_hyperparameters.html
freq = 'D'
prediction_length = 30
context_length = 30

**Defini??o de hiperpar?metros**: Configura o DeepAR com frequ?ncia di?ria e 10 ?pocas de treinamento, al?m de outros par?metros de contexto.

In [ ]:
hyperparametros = {
    'time_freq': freq,
    'epochs': "10",
    'context_length': str(context_length),
    'prediction_length': str(prediction_length)
}

In [ ]:
deepar.set_hyperparameters(**hyperparametros)

### Treinamento do modelo

O treinamento ? iniciado com os canais de dados de treinamento e teste.

In [ ]:
data_channels = {'train': s3_train_data, 'test': s3_test_data}
data_channels

In [ ]:
deepar.fit(inputs = data_channels)

### Deploy e previs?es

In [ ]:
job_name = deepar.latest_training_job.name
print('Job name: ', job_name)

**Cria??o do endpoint**: Um endpoint ? criado para que o modelo treinado possa fazer previs?es.

In [ ]:
endpoint = session.endpoint_from_job(job_name = job_name,
                                     initial_instance_count = 1,
                                     instance_type = "ml.m5.large",
                                     image_uri = container,
                                     role = role)

In [ ]:
print('Endpoint: ', endpoint)

**Classe DeepARPredictor**: Implementa fun??es para gerenciar o processo de predi??o usando o endpoint.

In [ ]:
class DeepARPredictor(sagemaker.predictor.Predictor):

    def set_prediction_parameters(self, freq, prediction_length):
        """Set the time frequency and prediction length parameters.
          This method **must** be called before being able to use `predict`.

        Parameters:
        freq -- string indicating the time frequency
        prediction_length -- integer, number of predicted time points

        Return value: none.
        """
        self.freq = freq
        self.prediction_length = prediction_length

    def predict(self, ts, cat=None, dynamic_feat=None,
                encoding="utf-8", num_samples=100, quantiles=["0.1", "0.5", "0.9"]):
        """Requests the prediction of for the time series listed in `ts`,
           each with the (optional) corresponding category listed in `cat`.

        Parameters:
        ts -- list of `pandas.Series` objects, the time series to predict
        cat -- list of integers (default: None)
        encoding -- string, encoding to use for the request (default: "utf-8")
        num_samples -- integer, number of samples to compute at prediction time (default: 100)
        quantiles -- list of strings specifying the quantiles to compute (default: ["0.1", "0.5", "0.9"])

        Return value: list of `pandas.DataFrame` objects, each containing the predictions
        """
        prediction_times = [x.index[-1] + datetime.timedelta(hours=1) for x in ts]

        req = self.__encode_request(ts, cat, dynamic_feat, encoding, num_samples, quantiles)
        res = super(DeepARPredictor, self).predict(req)
        return self.__decode_response(res, prediction_times, encoding)

    def __encode_request(self, ts, cat, dynamic_feat, encoding, num_samples, quantiles):

        instances = [series_to_obj(ts[k]) for k in range(len(ts))]

        configuration = {"num_samples": num_samples, "output_types": ["quantiles"], "quantiles": quantiles}
        http_request_data = {"instances": instances, "configuration": configuration}
        return json.dumps(http_request_data).encode(encoding)

    def __decode_response(self, response, prediction_times, encoding):
        response_data = json.loads(response.decode(encoding))
        list_of_df = []
        for k in range(len(prediction_times)):
            prediction_index = pd.date_range(start=prediction_times[k], freq=self.freq, periods=self.prediction_length)
            list_of_df.append(pd.DataFrame(data=response_data['predictions'][k]['quantiles'], index=prediction_index))
        return list_of_df

**Configura??o do preditor**: Um objeto DeepARPredictor ? criado para acessar o endpoint gerado.

In [ ]:
predictor = DeepARPredictor(endpoint_name = 'deepar-petr-2024-09-30-19-29-46-391',
                            sagemaker_session = session)
predictor.set_prediction_parameters(freq, prediction_length)
predictor.serializer.content_type = "application/json"

**Fazendo previs?es**: O modelo realiza previs?es usando os dados de treinamento.

In [ ]:
import datetime
previsoes = predictor.predict(serie_temporal_treinamento)

In [ ]:
len(previsoes[0])

In [ ]:
previsoes

**Visualiza??o das previs?es**: Um gr?fico ? gerado comparando as previs?es (mediana) com os dados reais do teste, al?m de incluir intervalos de confian?a entre os quantis 10% e 90%.

In [ ]:
for k in range(len(previsoes)):
    plt.figure(figsize=(12,6))
    previsoes[k]['0.5'].plot(label = 'Mediana', color = 'red')
    #serie_temporal_teste[k].plot(label = 'Alvo')
    serie_temporal_teste[k][-prediction_length - context_length:].plot(label = 'Alvo')
    p10 = previsoes[k]['0.1']
    p90 = previsoes[k]['0.9']
    plt.fill_between(p10.index, p10, p90, color = 'y', alpha = 0.5, label = 'Intervalo de confian?a')
    plt.legend()
    plt.show()